In [1]:
import duckdb
import pandas as pd

In [2]:
def load_dimensional_model():
    con = duckdb.connect('../loadsmart/dev.duckdb', read_only=True)
    fact = con.execute("SELECT * FROM fact_loadsmart").fetchdf()
    shippers = con.execute("SELECT * FROM dim_shippers").fetchdf()
    carriers = con.execute("SELECT * FROM dim_carriers").fetchdf()
    return fact, shippers, carriers


def delivered_in_month(fact, shippers, carriers, year, month):
    fact = fact[fact["load_was_cancelled"] == False]
    fact = fact[fact["delivery_date"].dt.strftime("%Y-%m") == f"{year}-{month:02d}"]
    fact = fact.merge(shippers, on="shipper_key").merge(carriers, on="carrier_key")
    return fact[["loadsmart_id", "shipper_name", "delivery_date", "pickup_city",
                 "pickup_state", "delivery_city", "delivery_state", "book_price", "carrier_name"]]

In [3]:
fact, shippers, carriers = load_dimensional_model()
export_df = delivered_in_month(fact, shippers, carriers, year=2025, month=3)
export_df

,loadsmart_id,shipper_name,delivery_date,pickup_city,pickup_state,delivery_city,delivery_state,book_price,carrier_name
0,206665369,Shipper 758,2025-03-15 13:50:00,Lodi,CA,Pacific,WA,1955.56,Carrier 86454


In [4]:
export_df.to_csv("delivered_loads_last_month.csv", index=False)